# 04 - Stability and Selection

Computes bootstrap-ARI stability and selects the winning clustering configuration. Defaults are intentionally light for notebook use; increase `BOOTSTRAP_B` for final results.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.express as px
from src import features, ingest, multi_config, preprocess, selector, stability

PRESENTATION = ("AAA", "2014J")
SAMPLE_N = 300
BOOTSTRAP_B = 2  # use 30 for final submission numbers
N_JOBS = 1

In [ ]:
tables = ingest.run(*PRESENTATION)
X = features.run(tables)
if len(X) > SAMPLE_N:
    X = X.sample(SAMPLE_N, random_state=42).sort_values("id_student").reset_index(drop=True)
ids, X_scaled, *_rest, columns, clean_features = preprocess.preprocess(X)
metrics_df, labels_by_config, reductions = multi_config.run_all(X_scaled)

In [ ]:
stability_df = stability.run_all(X_scaled, B=BOOTSTRAP_B, n_jobs=N_JOBS)
stability_df[["config_id", "reducer", "clusterer", "bootstrap_ari_mean", "bootstrap_ari_std"]]

In [ ]:
winner, ranked = selector.select_winning(metrics_df, stability_df)
winner[["config_id", "reducer", "clusterer", "k", "silhouette", "bootstrap_ari_mean", "composite"]]

In [ ]:
joined = metrics_df.drop(columns=["labels"], errors="ignore").merge(
    stability_df[["config_id", "bootstrap_ari_mean"]], on="config_id", how="left"
)
primary = joined[joined["clusterer"].ne("hdbscan")]
fig = px.scatter(
    primary,
    x="silhouette",
    y="bootstrap_ari_mean",
    color="clusterer",
    symbol="reducer",
    size="k",
    hover_name="config_id",
    title="Validity vs Stability",
)
fig.add_hline(y=0.40, line_dash="dot")
fig.show()

For the report, rerun this notebook with `BOOTSTRAP_B = 30` or run the full pipeline command from the README.